<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
## Why an VAE?


Implement a sequence-to-sequence VAE for SMILES: encoder, to latent vector, to decoder
- train with reconstruction loss + KL divergence

- sample new molecules from latent space

- explore latent interprolation between molecules

- evaliuate validity, uniqueness and novelty

- Show what a latent space is and how it enables controlled molecular generation: **latent space** good at exploration, interpolation and optimization tasks. THis is dimensionality reduction: nearby points correspond to similar molecules (think of Diffusion map)
- Demonstrate confidence with a **probabilistic generative model** vs purely autoregressive model (RNN)


**Network structure**
- input SMILES, tokenize + embed -> Encoder GRU
- Hidden state, sample latent z
- Decoder GRU starts from z, generates sequence token by token
- Loss = reconstruction (CE)

</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### Variational Inference, approximate posterior and the ELBO
Assume $x$ indicates an observed data point, and that depends on a low-dim **latent variable** $z$. If $p_{\theta}(x, z)$ is the joint probability, then the probability of generating the observed data point is obtained by marginalizing over $z$, $$p_{\theta}(x) = \int dz p(z) p_{\theta}(x|z),$$ $p_{\theta}(x|z)$ being the likelihood and $p(z)$ being the prior. Ideally, one would like to sample from this probability, but the integral is intractable.

So, introduce a more tractable distribution $q_{\Phi}(z|x)$. One can show that the following variational bound holds
$$\log p_{\theta}(x) \geq \langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} - KL(q_{\Phi}(z|x) || p(z)),$$ which we call the **Evidence Lower Bound (ELBO)**, and that $$\log p_{\theta}(x) = ELBO + KL\left( q_{\Phi}(z|x) || p_{\theta}(x|z)\right), $$ where the posterior has been evidenced. Please note that when $q_{\Phi}(z|x) = p_{\theta}(x|z)$, then **ELBO** approximates exactly the data probability. All the more reasons to optimize the parameters of the distributions in such a way that **ELBO** is maximal.

One usually selects a convenient prior $p(z)$ and also a convenient auxiliary function $q_{\Phi}$, and take it from there. 

So, if $p(z) = \mathcal{N}(0,1)$ and $q_{\Phi}(z|x) = \mathcal{N}(z; \mu_{\Phi}(x), \sigma_{\Phi}^2(x))$, the KL term in the ELBO equation can be computed exactly: $$KL(q_{\Phi}(z|x) || p(z)) = -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 -\mu_j^2 -\sigma^2_j \right)$$

**Takeaway:** The ELBO provides a tractable surrogate objective that balances two forces:  
- reconstruction accuracy via $\mathbb{E}_{q_\phi}[\log p_\theta(x|z)] $ 
- latent regularization via the KL divergence.

</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### Neural Network Parameterization

The core idea of the VAE is to represent the distributions in the ELBO as neural networks and optimize their parameters so that the ELBO is maximized, giving a tractable approximation to $\log p_\theta(x)$. After training, we can sample from the model by first drawing $z \sim p(z)$ and then generating a sequence from $p_\theta(x|z)$.

The network has two main components:

- **Encoder** $q_\phi(z|x)$:  
  Approximates the posterior distribution parameters over latent codes given an input sequence $x$.  
  For SMILES, this means mapping a full string into the parameters $\mu(x), \sigma(x)$ of a Gaussian distribution in latent space.

  The encoder goes first naturally, because we begin from observed data $x$

- **Decoder** $p_\theta(x|z)$:  
  Models the likelihood of the sequence conditioned on a latent variable $z$.  
  For SMILES, the decoder takes as input the latent code $z$ *and* the prefix tokens $(x_1, \dots, x_{T-1})$, and predicts the next tokens $(x_2, \dots, x_T)$.  


### A key insight
The decoder is structurally equivalent to a **standard RNN language model**, with one important extension: it is conditioned on a global latent fingerprint $z$. Since SMILES are sequential data, RNNs (or their modern replacements like Transformers) are a natural choice for this autoregressive modeling.

---
### Note
A network represents a probability distribution, in the sense that the network outputs parameters of a distribution object that we can sample from (here: Gaussian, for the encoder, and categorical = discrete random variable that can take one of $K$ possible values, for the decoder). In the SMILES decoder, every softmax output is exactly that — a categorical distribution over the next possible token.

The network itself is just a function approximator, but we force its output into a probabilitstic form. If we only output deterministc numbers, then we'd have a standard autoencoder: a fancy compressor that can memorize and reproduce, but not generate

</div>

In [1]:
import numpy as np, pandas as pd, os, math, random, time
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, QED

import matplotlib
%matplotlib inline

import matplotlib.pyplot as plt

import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [15]:
pad_length = 120     # this will have to be tuned according to the SMILES that we have
print('Max length of input SMILES = ' + str(pad_length))


# NN TRAINING
train_fraction = 0.8
val_fraction = 0.1

batch_size = 128
print('Batch size = ' + str(batch_size))

Max length of input SMILES = 120
Batch size = 128


In [3]:
def define_tokens(smiles: str):
    
    """ Takes in a SMILES string and returns the tokenization = split string into tokens """
    
    tokens = []
    i = 0
    while i < len(smiles):
        # check two-character tokens
        if smiles[i:i+2] in ("Cl", "Br", "@@"):
            tokens.append(smiles[i:i+2])
            i += 2
        else:
            tokens.append(smiles[i])
            i += 1
    return tokens

def encode(smiles):
    
    """ Tokenization: from tokens to sequence of integers """
    
    tokens = ["<START>"] + define_tokens(smiles) + ["<END>"]
    unk_id = stoi["<UNK>"]
    return [stoi.get(t, unk_id) for t in tokens]

def decode(id_list):
    
    """ De-tokenization: from sequence of integers to tokens """   
    
    tokens = [itos[i] for i in id_list if itos[i] not in ("<PAD>", "<START>", "<END>")]
    return "".join(tokens)

In [2]:
datafile = '250k_rndm_zinc_drugs_clean_3.csv'

input_data = os.path.join('data', datafile)

# Load CSV
df = pd.read_csv(input_data)
print(df.columns)    # printing the head
      
# Grab 'smiles' column
smiles_list = df["smiles"].dropna().tolist()

# Quick validity check with RDKit
valid_smiles = []
for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        valid_smiles.append(Chem.MolToSmiles(mol, canonical=True))     # make sure there are no duplicates

print(f"We now select the first 100,000 valid SMILES molecules, to be used for training")
random.shuffle(valid_smiles)
subset = valid_smiles[:100000]

Index(['smiles', 'logP', 'qed', 'SAS'], dtype='object')
We now select the first 100,000 valid SMILES molecules, to be used for training


In [6]:
def make_input_target(encoded):
    
    """
    Generate the encoder input, decoder input and target sequences, 
    from an encoded SMILES string
    """
    
    enc_in = encoded[:]     # input to ENCODER sees full sequence, to map to latent space
    dec_in = encoded[:-1]   #  (part of ) input to DECODER start with (<START, x_1, ... , x_{T}) (together with the latent z)
    tgt    = encoded[1:]       # target for DECODER training (x_1, ... , x_{T}, <END>)
    return enc_in, dec_in, tgt

# Build padded arrays for each split
def pad_batch(triplets, pad_id):
    
    """
    Convert a batch of (encoder, decoder, target) token sequences into
    padded NumPy arrays with consistent lengths.

    Each element of `triplets` is expected to be a tuple of three lists:
        - encoder input sequence (list of int tokens)
        - decoder input sequence (list of int tokens, e.g. prefix with <START>)
        - target sequence (list of int tokens, e.g. shifted outputs with <END>)

    This function:
      1. Finds the maximum sequence length in the encoder side (T)
         and in the decoder/target side (Td).
      2. Pads all sequences in the batch to those lengths using the given `pad_id`.
      3. Returns rectangular NumPy arrays suitable for conversion into tensors.

    Args:
        triplets (list of tuples): Batch of (encoder, decoder, target) sequences.
        pad_id (int): Token ID used for padding shorter sequences.

    Returns:
        enc_X (np.ndarray): Padded encoder input array of shape (B, T),
                            where B = batch size and T = max encoder length.
        dec_X (np.ndarray): Padded decoder input array of shape (B, Td),
                            where Td = max decoder length.
        Y     (np.ndarray): Padded target array of shape (B, Td).

    Notes:
        - Padding ensures all rows have the same length, which is required
          for batching in PyTorch/TensorFlow.
        - Standard naming convention: X = input, Y = output/labels.
    """
    
    encs = [np.array(t[0], dtype=np.int64) for t in triplets]
    decs = [np.array(t[1], dtype=np.int64) for t in triplets]
    tars = [np.array(t[2], dtype=np.int64) for t in triplets]
    T = max(map(len, encs))     #?
    Td = max(map(len, decs))    #?
    B = len(triplets)           # number of samples in batch

    def pad_to(arrs, Twant):
        X = np.full((B, Twant), pad_id, np.int64)
        for i,a in enumerate(arrs):
            X[i,:len(a)] = a
        return X

    enc_X = pad_to(encs, T)      # (B,T)
    dec_X = pad_to(decs, Td)     # (B,Td)
    Y     = pad_to(tars, Td)     # (B,Td)
    
    return enc_X, dec_X, Y       # standard notation: X is input, Y is output

In [7]:
# --- User-defined SMILES vocabulary/alphabet: ideally, one could build this from analyzing the strings ---

VOCAB = [
    # Special tokens [padding, start, end, unknown symbol]
    "<PAD>", "<START>", "<END>", "<UNK>",

    # Common atoms
    "C", "O", "N", "S", "P", "F", "Cl", "Br", "I",

    # Aromatic atoms (lowercase form often used in SMILES)
    "c", "o", "n", "s", "p",

    # Bonds
    "-", "=", "#",

    # Branching / rings
    "(", ")", "[", "]",

    # Ring closure digits
    "1","2","3","4","5","6","7","8","9",
    
    # Stereo / charges
    "@", "@@", "+", "H"
]


# Build dictionaries: use increasing integer numbers for tokenization
stoi = {tok: i for i, tok in enumerate(VOCAB)}   # from tokens to integers
itos = {i: tok for tok, i in stoi.items()}     # from integers to tokens

V        = len(stoi)    # dictionary size
print('Dictionary length for tokenization = ' + str(V))

Dictionary length for tokenization = 38


In [14]:
# integer corresponding to the padding entry
pad_id = stoi["<PAD>"]

print('Prepare input nd output arrays')
test_smiles = ["CCO", "CC(=O)O", "HCl"]

triplets = [make_input_target(encode(s)) for s in test_smiles]
enc_X, dec_X, Y = pad_batch(triplets, pad_id)

print(enc_X)
print(dec_X)
print(Y)

Prepare input nd output arrays
[[ 1  4  4  5  2  0  0  0  0]
 [ 1  4  4 21 19  5 22  5  2]
 [ 1 37 10  2  0  0  0  0  0]]
[[ 1  4  4  5  0  0  0  0]
 [ 1  4  4 21 19  5 22  5]
 [ 1 37 10  0  0  0  0  0]]
[[ 4  4  5  2  0  0  0  0]
 [ 4  4 21 19  5 22  5  2]
 [37 10  2  0  0  0  0  0]]


In [16]:
dataset = TensorDataset(torch.from_numpy(enc_X), torch.from_numpy(dec_X), torch.from_numpy(Y))
N = len(dataset)

n_train = int(train_fraction * N)
n_val   = int(val_fraction * N)
n_test  = N - n_train - n_val

print(f"train={n_train}, val={n_val}, test={n_test}")
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],generator=torch.Generator().manual_seed(42))  # reproducible split

# data loader
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size)
test_loader  = DataLoader(test_ds, batch_size=batch_size)

print(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

train=2, val=0, test=1
2 0 1


<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

### VAE loss for SMILES Generation
As discussed already, the loss function here is $\mathcal{L}(x) = \langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} - KL(q_{\Phi}(z|x) || p(z))$ and we look to maximize that with respect to network parameters $(\Phi, \theta)$. 

- **KL term:** The KL term can be computed exactly from the assumption that $p(z)$ and $q_{\Phi}$ are Gaussians.

- **Reconstruction term:** What about the reconstruction term? $\langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} = \int dz q_{\Phi}(z|x) \log p_{\theta}(x|z)$, which is totally impractical. We can then use the finite sum approximation/Monte Carlo approach here, and draw $z$ samples from $q_{\Phi}$:
$$\langle \log p_{\theta}(x|z)\rangle_{q_{\Phi}} \approx \frac{1}{N}\sum_{i=1}^N \log p_{\theta}(x|z_i), \quad z_i \sim q_{\Phi} $$

So that
$$\mathcal{L}(x) = \frac{1}{N}\sum_{i=1}^N \log p_{\theta}(x|z_i \sim q_{\Phi}) -\frac{1}{2} \sum_j \left(1 + \log \sigma_j^2 -\mu_j^2 -\sigma^2_j \right)$$

Now, for practical reasons, one chooses $N=1$ for each data-point, and we woudl still get an unbiased estimator of the integral

### autoregressive factorization for SMILES
Specifically for the SMILES generation problem now, $x_{<t} = (x_1, \ldots, x_{T-1})$ is an (encoded) prefix string, so that $$ p_{\theta}(x|z) = \prod_t p_{\theta}(x_t | x_{<t}, z),$$ so the probability of the next token is conditioned on the full prefix and the value of the latent variable $z$. 
- During training, $(y_1, \ldots, y_{T-1}) = (x_2, \ldots, x_{T})$ provides the targets.
- The decoder outputs logits at each step, which are softmaxed into probabilities
- The cross-entropy compares predicted distributions to the one-hot ground truth tokens, i.e. $\sum_t y_t \log p_{\theta}(x_t | x_{<t}, z)$

### Final perspective
Thus, the reconstruction term reduces to a **standard autoregressive cross-entropy loss**, while the KL term enforces that latent codes \(z\) live in a smooth Gaussian latent space. This is what makes the model **generative** rather than just memorizing training sequences.

Note that, differently from a vanilla autoencoder, we are not optimizing a deterministic error, but optimize a stochastic objective (Monte-Carlo estimate of ELBO)

</div>

In [ ]:
def kl_gaussian(mu, logvar):
    """ 
    Compute the KL divergence between the Gaussian encoder and the Gaussian latent prior
    KL(N(mu, diag(exp(logvar))) || N(0,I)) per sample = -0.5 * sum(1 + logσ² - μ² - σ²)
    """
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1)  # (B,)

def vae_loss(logits, y_target, pad_id, mu, logvar, beta=1.0):

    """
    VAE loss = reconstruction cross-entropy (masked over PAD) + beta * KL(q||p).

    Args:
        logits   (FloatTensor): (B, T, V) decoder logits.
        y_target (LongTensor):  (B, T)   target token ids (PAD included).
        pad_id   (int):         padding token id to ignore in CE.
        mu       (FloatTensor): (B, z_dim) encoder mean.
        logvar   (FloatTensor): (B, z_dim) encoder log-variance.
        beta     (float):       KL weight (for β-VAE / KL warm-up).

    Returns:
        loss        (Tensor): scalar total loss for backprop.
        recon_mean  (float):  mean reconstruction CE (per token, per sample).
        kl_mean     (float):  mean KL divergence.
    """
    
    B,T,V = logits.shape
    
    # Reconstruction term: token-level cross-entropy (ignore PAD)
    ce = F.cross_entropy(logits.view(-1, V), y_target.reshape(-1),
                         ignore_index=pad_id, reduction='none'      # there will be some reduction to a scalar at some point
                        ).view(B, T)
    
    # mask out PAD positions fully
    mask = (y_target != pad_id).float()
    recon = (ce * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)  # per-sample CE, so that long smiles do not dominate
    
    # KL term per sample
    kl = kl_gaussian(mu, logvar)  # (B,)
    # Total (mean over batch): this is the EMLB averaged over the batch
    loss = recon.mean() + beta * kl.mean()
    
    return loss, recon.mean().item(), kl.mean().item()

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

Architecture of our Variational Autoencoder (VAE)

- **Embedding Layer**: Maps token IDs (integers) into dense vectors (=continuous values).
- **Encoder GRU**: Processes the embedded SMILES sequence and outputs a 
  fixed-length hidden representation summarizing the molecule.
- **Latent Distribution Heads**: Two linear layers projecting the encoder 
  hidden state into the mean (mu) and log-variance (logvar) of the 
  approximate posterior distribution $q(z|x)$.
- **Reparameterization Trick**: Samples latent vectors $z$ from $q(z|x)$ 
  in a differentiable way, enabling end-to-end training with backpropagation.
- **Decoder GRU**: Reconstructs SMILES sequences by autoregressively 
  predicting the next token, conditioned on $z$ and the prefix of known tokens 
  (teacher forcing during training).
- **Projection Layer**: Maps decoder hidden states back into the vocabulary 
  space, producing logits over possible next tokens, which are then softmaxed and enter the cross entropy

Training Objective:

The model is optimized via the **Evidence Lower Bound (ELBO)**, consisting of:
- Reconstruction Loss: Cross-entropy between predicted tokens and ground truth.
- KL Divergence: Regularizes $q(z|x)$ to be close to the prior $p(z) = \mathcal{N}(0,I)$.

</div>

Teacher forcing = during training, feed the decoder the true previous tokens instead of its own predictions

In [ ]:
class SmilesVAE(nn.Module):

       """
    Variational Autoencoder (VAE) for molecular generation using SMILES strings.

    This model learns a probabilistic latent-space representation of molecules 
    from tokenized SMILES sequences.

    Args:
        vocab_size (int): Number of tokens in the SMILES vocabulary.
        emb_dim (int): Dimension of token embeddings.
        enc_h (int): Hidden size of the encoder GRU.
        dec_h (int): Hidden size of the decoder GRU.
        z_dim (int): Dimensionality of the latent space.
        pad_id (int): Token ID reserved for padding (ignored in loss).
    
    Inputs:
        enc_in (LongTensor): Tensor of shape (B, T) with tokenized SMILES input.
        dec_in (LongTensor): Tensor of shape (B, T) with shifted input tokens 
                             for teacher forcing.
    
    Outputs:
        logits (FloatTensor): Predicted unnormalized scores of shape (B, T, V) 
                              over the vocabulary.
        mu (FloatTensor): Mean of the latent distribution, shape (B, z_dim).
        logvar (FloatTensor): Log-variance of the latent distribution, shape (B, z_dim).

    Example:
        >>> model = SmilesVAE(vocab_size=40, emb_dim=128, enc_h=256, dec_h=256, z_dim=64, pad_id=0)
        >>> logits, mu, logvar = model(enc_in, dec_in)
        >>> loss = vae_loss_fn(logits, target, mu, logvar, pad_id=0)
    """
    
    def __init__(self, vocab_size, pad_id=0, emb_dim=128, hidden_dim=256, num_layers=1, dropout=0.0, use_lstm=False, tie_weights=True):
        super().__init__()
        
        self.pad_id = pad_id
        self.vocab_size = vocab_size

        # here we list the building blocks we will need later
        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)     #embedding layer, pad_id ignored during training
                                                                               # turn each token ID (int) into a continuous vector of size `emb_dim`
        # ---- Encoder (GRU) routines ----
        self.enc_rnn = rnn_GRU(       # this is the actual RNN that will process the sequences of tokens at each time step
            emb_dim, enc_h, num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        # reparametrization
        self.to_mu     = nn.Linear(enc_h, z_dim)    # just one mapping from hidden size to latent size
        self.to_logvar = nn.Linear(enc_h, z_dim)

        # --- Decoder (GRU) routines ---- 
        self.z_to_h0 = nn.Linear(z_dim, dec_h * num_layers)    # need to go from latent variable z to hidden state
        self.dec_rnn = rnn_GRU(       # this is the actual RNN that will process the sequences of tokens at each time step
            dec_dim, dec_h, num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        # from hidden state to logits
        self.proj = nn.Linear(dec_dim, vocab_size, bias=True)
        
        # small initialization
        for m in [self.to_mu, self.to_logvar, self.z_to_h0, self.proj]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
        
    def encode(self, x):
        """ Start out with the data, encode that, and compute the parameters of the approx Gaussian posterior """
        x_emb = self.embed(x)
        _, hN = self.enc_rnn(x_emb)     # extract the final hidden state [hiddens stae for every token, last hidden state for each sequence]
        h_last = hN[-1]                 # the final hidden state is a summary of the whole molecule

        mu, logvar = self.to_mu(h_last), self.to_logvar(h_last)       # take hidden state and project to two vectors
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """
        z ~ N(mu, diag(exp(logvar))): latent fingerprint of the molecle, using a trick to make gradients flow
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std       # this makes sampling differentiable so gradients flow / 

    def init_dec_hidden(self, z, num_layers=1, dec_h=None):
        """
        Initialize hidden state from latent variabile z
        Map z -> initial hidden state for decoder GRU.
        Returns h0 of shape (num_layers, B, dec_h)
        """
        B = z.size(0)
        dec_h = dec_h or (self.dec_rnn.hidden_size)
        h0_flat = torch.tanh(self.z_to_h0(z))          # (B, dec_h*num_layers)
        h0 = h0_flat.view(B, num_layers, dec_h).permute(1,0,2).contiguous()
        return h0
        
    def decode(self, y_in, h0):   # takes in the string (x) & the hidden state (z)
        """

        Take in y_in and the hidden state h0 from the latent state, and decode back into the logits.
        Again, leaving out the latent injecting, this is just another RNN, like that we used in the RNN_SMILES notebook
        
        y_in: (B, T) tokens (teacher-forced inputs, start with <START>)
        h0:   (num_layers, B, dec_h) initial hidden
        returns logits: (B, T, V)
        """
        y_emb = self.emb(y_in)     
        y, _ = self.dec_rnn(y_emb, h0)      # take the full story here, (B, T, V) logits, not just the last step, conditioned on the 
                                            # last hidden state
        logits = self.proj(y)
        return logits
        
    def forward(self, x_in, y_in):    # encoder input and decoder input

        mu, logvar = self.encode(x_in)
        z = self.reparameterize(mu, logvar)     # from `x_in` to its `z` latent mapping 

        h0 = self.init_dec_hidden(z, num_layers=self.dec_rnn.num_layers,
                                  dec_h=self.dec_rnn.hidden_size)      # we need to know the hidden state associated with `z`, so to 
                                                                       # decode
        
        logits = self.decode(y_in, h0)     # takes `y_in` AND `z` (via the hidden state)

        return logits, mu, logvar     # one normalized score per vocab token, ready for cross-entropy and KL loss function

In [ ]:
model = SmilesVAE(vocab_size, pad_id, emb_dim=128, enc_h=256, dec_h=256, z_dim=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

def run_epoch(loader, train=True, beta=1.0, clip=1.0):
    
    model.train(train)
    # initialize temporary variables for saving losses
    total_loss, total_recon, total_kl, n = 0.0, 0.0, 0.0, 0
    
    for enc_X, dec_X, Y in loader:
        enc_X, dec_X, Y = enc_X.to(device), dec_X.to(device), Y.to(device)
        if train: 
            opt.zero_grad()     # set all the grafients to zero
            
        logits, mu, logvar = model(enc_X, dec_X)               # (B,T,V), (B,z), (B,z)
        loss, rce, kl = vae_loss(logits, Y, pad_id, mu, logvar, beta=beta)
        
        if train:
            loss.backward()      # backpropagate and take a step of ADAM 
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            
        bs = enc_X.size(0)
        total_loss  += loss.item() * bs
        total_recon += rce * bs
        total_kl    += kl * bs
        n += bs
    return total_loss/n, total_recon/n, total_kl/n

epochs = 20
for ep in range(1, epochs+1):
    beta = min(1.0, ep / 10)  # warm up over 10 epochs (suggested by chatgpt)
    tr = run_epoch(train_loader, train=True, beta=beta)
    va = run_epoch(val_loader,   train=False, beta=beta)
    print(f"epoch {ep:02d} | beta {beta:.2f} | "
          f"train loss {tr[0]:.3f} (recon {tr[1]:.3f}, KL {tr[2]:.3f}) | "
          f"val loss {va[0]:.3f} (recon {va[1]:.3f}, KL {va[2]:.3f})")


<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">

### Use trained decoder to generate samples

Sampling new points requires sampling from $p_\theta (x) = \int p(z) p_{\theta}(x|z) dz$. We will sample from the latent space $z \sim \mathcal{N}(0,1)$ and decode.

One does not have to compute the marginal $p_\theta(x)$ explicitly. Instead, sample "ancestrally" along the factorization of the joint probability $p_\theta(x,z) = p(z) p_\theta(x|z)$, which is the generative story. The marginal is defined as the integral of the joint probability. When we ignore $z$ after the sampling, we are effectively sampling from the marginal (marginalization comes for free by ignoring latent variables).

So, sampling from the marginal $p_\theta(x)$ involves two steps:

- Sample a latent:  $x \sim p(z)$
- Sample a sequence autoregressively from the decoder $x \sim p_\theta(x|z) = \prod_t p_\theta (x_t | x_{<t}, z)$

</div>

In [ ]:
@torch.no_grad()
def decode_from_z(z, max_len=120, temperature=0.9, top_k=30):
    model.eval()
    B = z.size(0)
    h0 = model.init_dec_hidden(z, num_layers=model.dec_rnn.num_layers,
                               dec_h=model.dec_rnn.hidden_size)
    # start with <START>
    cur = torch.full((B,1), stoi["<START>"], dtype=torch.long, device=device)
    out_tokens = [[] for _ in range(B)]

    for _ in range(max_len):
        logits = model.decode(cur, h0)[:, -1, :]   # (B,V) last step
        logits = logits / max(1e-6, temperature)
        if top_k is not None:
            k = min(top_k, logits.size(-1))
            topv, topi = torch.topk(logits, k=k, dim=-1)
            probs = F.softmax(topv, dim=-1)
            idx = torch.multinomial(probs, 1)                # (B,1) index in top-k
            next_ids = topi.gather(-1, idx)                  # map back to vocab ids
        else:
            probs = F.softmax(logits, dim=-1)
            next_ids = torch.multinomial(probs, 1)
        for i in range(B):
            out_tokens[i].append(next_ids[i,0].item())
        cur = torch.cat([cur, next_ids], dim=1)
        # early stop if all produced <END>
        if all(out_tokens[i][-1] == stoi["<END>"] for i in range(B)):
            break
    # convert to SMILES
    def ids_to_smiles(ids):
        toks = [itos[i] for i in ids if i not in (stoi["<PAD>"], stoi["<START>"], stoi["<END>"])]
        return "".join(toks)
    smiles = [ids_to_smiles(seq) for seq in out_tokens]
    return smiles